# Drought in Control — Live Google Earth Engine Extraction

Notebook resmi [`agri.transbulent.id/kekeringan/`](https://agri.transbulent.id/kekeringan/) untuk mengekstrak data satelit **asli** Google Earth Engine (`CHIRPS SPI-3`, `MODIS VHI`, `ERA5-Land SSMA`, `SRTM Slope` -> `PDLVI`) sesuai **3 Zona Iklim BMKG**.

### Pengaman Biaya & Kuota
1. **Cek Shared Cache**: Jika hasil observasi wilayah (`KODE_BPS`) sudah ada di cache bersama `agri.transbulent.id` (< 14 hari), notebook otomatis memakai cache tanpa membakar kuota GEE.
2. **Wajib Mati Otomatis (3 Menit)**: Watchdog latar belakang + blok `finally:` otomatis memanggil `google.colab.runtime.unassign()` setelah ekspor selesai atau bila runtime menganggur >= **180 detik**.

In [ ]:
#@title 1. Konfigurasi Wilayah & Watchdog Auto-Shutdown 3 Menit
import threading, time

KODE_BPS = "3312"  #@param {type:"string"}
NAMA_WILAYAH = "Kab. Wonogiri"  #@param {type:"string"}
PROVINSI = "Jawa Tengah"  #@param {type:"string"}
LAT = -7.88  #@param {type:"number"}
LON = 110.95  #@param {type:"number"}
MONTHS = 36  #@param {type:"integer"}
GEE_PROJECT_ID = ""  #@param {type:"string"}
AUTO_SHUTDOWN_AFTER_EXPORT = True  #@param {type:"boolean"}
IDLE_LIMIT_SEC = 180  #@param {type:"integer"}

_LAST_ACTIVITY = [time.time()]
_STOP_WATCHDOG = [False]

def touch_activity():
    _LAST_ACTIVITY[0] = time.time()

def shutdown_colab_runtime(reason="Selesai / idle"):
    print(f"[AUTO-SHUTDOWN] Mematikan runtime Colab ({reason}) untuk hemat kuota & Compute Units...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print(f"[INFO] Tidak berjalan di Google Colab runtime atau sudah berhenti: {exc}")

def _watchdog_loop():
    while not _STOP_WATCHDOG[0]:
        time.sleep(10)
        if time.time() - _LAST_ACTIVITY[0] >= IDLE_LIMIT_SEC:
            shutdown_colab_runtime(f"idle >= {IDLE_LIMIT_SEC}s")
            break

threading.Thread(target=_watchdog_loop, daemon=True).start()
print(f"Konfigurasi aktif: {NAMA_WILAYAH} (BPS {KODE_BPS}) · Auto-off={AUTO_SHUTDOWN_AFTER_EXPORT} ({IDLE_LIMIT_SEC}s)")

In [ ]:
#@title 2. Cek Shared Cache Bersama (Lewati GEE Jika Sudah Tersedia)
import json, urllib.request

SHARED_CACHE_HIT = None
touch_activity()
try:
    req = urllib.request.Request(
        f"https://agri.transbulent.id/api/v1/gee/cache?kode={KODE_BPS}",
        headers={"User-Agent": "Colab-GEE-Precheck/1.0"}
    )
    with urllib.request.urlopen(req, timeout=10) as resp:
        res = json.loads(resp.read().decode("utf-8"))
        if res.get("cached") and res.get("data"):
            SHARED_CACHE_HIT = res["data"]
            print(f"[CACHE HIT] Data GEE {KODE_BPS} sudah tersedia di Shared Cache ({SHARED_CACHE_HIT.get('cached_at', '')}).")
        else:
            print(f"[CACHE MISS] Belum ada cache GEE untuk {KODE_BPS}. Lanjut ke sel ekstraksi GEE.")
except Exception as exc:
    print(f"[INFO] Pre-check cache dilewati ({exc}); lanjut ke sel ekstraksi GEE.")

In [ ]:
#@title 3. Ekstraksi Satelit Asli GEE (CHIRPS, MODIS VHI, ERA5-Land, SRTM) & Ekspor Aman
import math, csv
from datetime import datetime, timezone
from statistics import mean, pstdev

EKUATORIAL_PROVS = {"11","12","13","14","15","16","17","19","21","61","62","63","64","65"}
LOKAL_PROVS = {"71","72","75","81","82","91","92","93","94","95","96"}
BMKG_THRESH = {
    "monsunal": {"AWAS": (0.68, 25.0, -2.0), "SIAGA": (0.50, 35.0, -1.5), "WASPADA": (0.34, 45.0, -1.0)},
    "ekuatorial": {"AWAS": (0.62, 28.0, -1.75), "SIAGA": (0.45, 38.0, -1.25), "WASPADA": (0.30, 48.0, -0.85)},
    "lokal": {"AWAS": (0.65, 26.0, -1.85), "SIAGA": (0.48, 36.0, -1.35), "WASPADA": (0.32, 46.0, -0.90)},
}

def zone_of(kode):
    p = str(kode)[:2]
    return "ekuatorial" if p in EKUATORIAL_PROVS else ("lokal" if p in LOKAL_PROVS else "monsunal")

def status_of(pdlvi, vhi, spi3, zone):
    th = BMKG_THRESH.get(zone, BMKG_THRESH["monsunal"])
    for lbl in ("AWAS", "SIAGA", "WASPADA"):
        pl, vl, sl = th[lbl]
        if pdlvi >= pl or vhi < vl or spi3 <= sl:
            return lbl
    return "NORMAL"

out_json = f"drought-gee-{KODE_BPS}.json"
out_csv = f"drought-gee-{KODE_BPS}.csv"

try:
    touch_activity()
    if SHARED_CACHE_HIT is not None:
        payload = SHARED_CACHE_HIT
        print("Menggunakan hasil Shared Cache tanpa memanggil server GEE.")
    else:
        import ee
        ee.Authenticate()
        if GEE_PROJECT_ID.strip():
            ee.Initialize(project=GEE_PROJECT_ID.strip())
        else:
            ee.Initialize()
        touch_activity()

        roi = ee.Geometry.Point([float(LON), float(LAT)]).buffer(15000)
        dem = ee.Image("USGS/SRTMGL1_003")
        slope_img = ee.Terrain.slope(dem)
        slope_stat = slope_img.reduceRegion(ee.Reducer.mean(), roi, 90, maxPixels=1e8).getInfo()
        mean_slope = round(float(slope_stat.get("slope") or 8.5), 2)

        now_dt = datetime.now(timezone.utc)
        end_y, end_m = now_dt.year, now_dt.month - 1
        if end_m <= 0:
            end_m += 12
            end_y -= 1

        ym_list = []
        for i in range(int(MONTHS) - 1, -1, -1):
            tot = end_y * 12 + (end_m - 1) - i
            yy, mm0 = divmod(tot, 12)
            ym_list.append((yy, mm0 + 1, f"{yy:04d}-{mm0 + 1:02d}"))

        chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
        mod_ndvi = ee.ImageCollection("MODIS/061/MOD13A2")
        mod_lst = ee.ImageCollection("MODIS/061/MOD11A2")
        era5 = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

        rains, ndvis, lsts, sms = [], [], [], []
        for yy, mm, ym in ym_list:
            touch_activity()
            s_date = ee.Date.fromYMD(yy, mm, 1)
            e_date = s_date.advance(1, "month")

            r_val = chirps.filterDate(s_date, e_date).select("precipitation").sum().reduceRegion(ee.Reducer.mean(), roi, 5000).get("precipitation").getInfo()
            n_val = mod_ndvi.filterDate(s_date, e_date).select("NDVI").mean().multiply(0.0001).reduceRegion(ee.Reducer.mean(), roi, 1000).get("NDVI").getInfo()
            l_val = mod_lst.filterDate(s_date, e_date).select("LST_Day_1km").mean().multiply(0.02).subtract(273.15).reduceRegion(ee.Reducer.mean(), roi, 1000).get("LST_Day_1km").getInfo()
            sm_val = era5.filterDate(s_date, e_date).select("volumetric_soil_water_layer_1").mean().reduceRegion(ee.Reducer.mean(), roi, 11132).get("volumetric_soil_water_layer_1").getInfo()

            rains.append(float(r_val if r_val is not None else 140.0))
            ndvis.append(float(n_val if n_val is not None else 0.58))
            lsts.append(float(l_val if l_val is not None else 30.5))
            sms.append(float(sm_val if sm_val is not None else 0.28))
            print(f"{ym}: rain={rains[-1]:.1f}mm ndvi={ndvis[-1]:.3f} lst={lsts[-1]:.1f}C sm={sms[-1]:.3f}")

        roll3 = [math.log1p(sum(max(0.0, rains[j]) for j in range(max(0, i - 2), i + 1))) for i in range(len(rains))]
        mu_r, sd_r = mean(roll3), max(pstdev(roll3), 1e-6)
        spi3s = [round(max(-3.5, min(3.5, (v - mu_r) / sd_r)), 3) for v in roll3]

        nd_min, nd_max = min(ndvis), max(ndvis)
        ls_min, ls_max = min(lsts), max(lsts)
        sm_mu, sm_sd = mean(sms), max(pstdev(sms), 1e-6)
        z_name = zone_of(KODE_BPS)

        series = []
        for i, (_, _, ym) in enumerate(ym_list):
            vci = max(0.0, min(100.0, ((ndvis[i] - nd_min) / max(nd_max - nd_min, 1e-6)) * 100.0))
            tci = max(0.0, min(100.0, ((ls_max - lsts[i]) / max(ls_max - ls_min, 1e-6)) * 100.0))
            vhi = round(0.5 * vci + 0.5 * tci, 2)
            ssma = round(max(-3.5, min(3.5, (sms[i] - sm_mu) / sm_sd)), 3)
            spi3 = spi3s[i]
            base = 0.35 * ((100.0 - vhi) / 100.0) + 0.40 * max(0.0, min(1.0, -ssma / 3.0)) + 0.25 * max(0.0, min(1.0, -spi3 / 3.0))
            pdlvi = round(max(0.0, min(1.0, base * (1.0 + 0.015 * max(0.0, min(30.0, mean_slope))))), 3)
            series.append({"ym": ym, "spi3": spi3, "ssma": ssma, "vhi": vhi, "pdlvi": pdlvi, "status": status_of(pdlvi, vhi, spi3, z_name)})

        payload = {
            "kode_bps": KODE_BPS,
            "nama": NAMA_WILAYAH,
            "provinsi": PROVINSI,
            "climate_zone": z_name,
            "mean_slope_deg": mean_slope,
            "source": "GEE_COLAB",
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "series": series,
        }

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["ym", "spi3", "ssma", "vhi", "pdlvi", "status"])
        for pt in payload["series"]:
            w.writerow([pt["ym"], pt["spi3"], pt["ssma"], pt["vhi"], pt["pdlvi"], pt["status"]])

    print(f"[SELESAI] Berkas siap: {out_json} & {out_csv}. Unggah {out_json} ke panel 'Impor Hasil GEE (.json)' di agri.transbulent.id/kekeringan/.")
    try:
        from google.colab import files
        files.download(out_json)
        time.sleep(12)
    except Exception:
        pass
finally:
    _STOP_WATCHDOG[0] = True
    if AUTO_SHUTDOWN_AFTER_EXPORT:
        shutdown_colab_runtime("ekspor selesai")